# Day 3 · 실습 2. LSTM 문장 분류와 Hidden State 활용

**연결 이론:** 이론 3의 LSTM, Cell State·Hidden State, 게이트에 의한 정보 갱신.

**이번 실습:** A형에서 LSTM 문장 의도 분류를 구현하고, B형 1단계에서 은닉 차원을 변경합니다. B형 2단계에서는 **요청 순서에 따라 최종 의도가 바뀌는 새로운 문장**을 대상으로 예측 함수를 직접 완성합니다.

**진행:** A형 18분 → B형 1단계 10분 → B형 2단계 20분 → 결과 확인 2분.

이 노트북과 `weights/` 폴더를 함께 둔 상태에서 실행하세요. CPU 환경에서도 실행할 수 있습니다.

## A형 · LSTM 문장 의도 분류 

`예약 / 취소 / 변경` 문장을 학습하고, LSTM의 `output`, `h_n`, `c_n`이 무엇을 나타내는지 확인합니다.

### A-1. 라이브러리 및 실행 설정

PyTorch를 불러오고 재현을 위해 시드를 고정합니다.

In [1]:
import torch
from torch import nn

# 수업용 소규모 데이터이므로 CPU에서 실행합니다.
torch.set_num_threads(1)
torch.manual_seed(42)
print("PyTorch:", torch.__version__)

PyTorch: 2.15.0+cpu


### A-2. 학습 문장과 정답 준비

같은 문장 형태에서 요청 단어만 달라지도록 데이터를 만들고, 단어를 정수 ID로 변환합니다.

In [2]:
# 정답 클래스: 문장 전체를 읽고 예약·취소·변경 중 하나를 예측합니다.
label_names_a = ["예약", "취소", "변경"]
label_to_id_a = {name: idx for idx, name in enumerate(label_names_a)}

sentences_a = []
targets_a = []
for intent in label_names_a:
    for day in ["오늘", "내일", "모레", "주말"]:
        for period in ["오전", "오후"]:
            for topic in ["시간", "일정"]:
                sentences_a.append(f"{intent} {day} {period} {topic} 부탁")
                targets_a.append(label_to_id_a[intent])

vocab_a = sorted({token for text in sentences_a for token in text.split()})
token_to_id_a = {token: idx for idx, token in enumerate(vocab_a)}

# 모든 문장이 5개 토큰으로 이루어져 있으므로 Padding 없이 묶을 수 있습니다.
x_a = torch.tensor(
    [[token_to_id_a[token] for token in text.split()] for text in sentences_a],
    dtype=torch.long,
)
y_a = torch.tensor(targets_a, dtype=torch.long)

print("학습 문장 수:", len(sentences_a))
print("첫 문장:", sentences_a[0], "→ 정답:", label_names_a[y_a[0].item()])
print("입력 Shape:", tuple(x_a.shape), "정답 Shape:", tuple(y_a.shape))

학습 문장 수: 48
첫 문장: 예약 오늘 오전 시간 부탁 → 정답: 예약
입력 Shape: (48, 5) 정답 Shape: (48,)


### A-3. LSTM 분류 모델 구현

`Embedding → LSTM → Linear`를 연결합니다. 문장 분류에는 마지막 Hidden State인 `h_n[-1]`을 사용합니다.

In [3]:
class LSTMIntentClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, embedding_dim: int = 12, hidden_size: int = 16):
        super().__init__()
        # 토큰 ID를 학습 가능한 단어 벡터로 변환합니다.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # batch_first=True: 입력 Shape은 (배치, 시점, 임베딩 차원)입니다.
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        # 문장 전체의 마지막 Hidden State를 클래스별 Logits로 변환합니다.
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, token_ids: torch.Tensor):
        embedded = self.embedding(token_ids)
        # output: 모든 시점의 Hidden State / h_n: 마지막 Hidden State
        # c_n: 마지막 Cell State (LSTM은 h_n과 c_n을 모두 반환합니다).
        output, (h_n, c_n) = self.lstm(embedded)
        logits = self.head(h_n[-1])  # 마지막 계층·마지막 시점의 Hidden State 활용
        return logits, output, h_n, c_n


model_a = LSTMIntentClassifier(
    vocab_size=len(vocab_a), num_classes=len(label_names_a),
    embedding_dim=12, hidden_size=16,
)
print(model_a)

LSTMIntentClassifier(
  (embedding): Embedding(12, 12)
  (lstm): LSTM(12, 16, batch_first=True)
  (head): Linear(in_features=16, out_features=3, bias=True)
)


### A-4. LSTM이 반환하는 세 가지 상태 확인

두 문장을 입력해 모든 시점의 출력(`output`)과 마지막 상태(`h_n`, `c_n`)의 Shape를 확인합니다.

In [4]:
model_a.eval()
with torch.no_grad():
    embedded = model_a.embedding(x_a[:2])
    output_a, (h_n_a, c_n_a) = model_a.lstm(embedded)
    logits_a = model_a.head(h_n_a[-1])

print("Embedding:", tuple(embedded.shape))
print("output (전체 시점):", tuple(output_a.shape))
print("h_n (마지막 Hidden State):", tuple(h_n_a.shape))
print("c_n (마지막 Cell State):", tuple(c_n_a.shape))
print("문장 분류 Logits:", tuple(logits_a.shape))

# 단일 계층·단방향 LSTM에서는 output의 마지막 시점과 h_n[-1]이 같습니다.
assert torch.allclose(output_a[:, -1, :], h_n_a[-1])
print("마지막 output = h_n[-1]:", True)

Embedding: (2, 5, 12)
output (전체 시점): (2, 5, 16)
h_n (마지막 Hidden State): (1, 2, 16)
c_n (마지막 Cell State): (1, 2, 16)
문장 분류 Logits: (2, 3)
마지막 output = h_n[-1]: True


### A-5. 문장 분류 모델 학습

LSTM이 반환한 마지막 Hidden State로 세 가지 요청 유형을 분류하도록 학습합니다.

In [5]:
optimizer_a = torch.optim.Adam(model_a.parameters(), lr=0.02)
criterion_a = nn.CrossEntropyLoss()

for epoch in range(1, 81):
    model_a.train()
    optimizer_a.zero_grad()  # 이전 반복에서 계산된 기울기를 초기화합니다.

    logits, _, _, _ = model_a(x_a)  # 정답 클래스마다 Logit 하나씩 계산합니다.
    loss = criterion_a(logits, y_a)  # CrossEntropyLoss에는 Softmax 이전 Logits를 넣습니다.
    loss.backward()                 # LSTM 내부 가중치까지 역전파합니다.
    optimizer_a.step()              # 계산된 기울기로 가중치를 갱신합니다.

    if epoch in [1, 20, 40, 60, 80]:
        print(f"epoch {epoch:>2} | loss = {loss.item():.4f}")

model_a.eval()
with torch.no_grad():
    logits, _, _, _ = model_a(x_a)
    train_accuracy = (logits.argmax(dim=-1) == y_a).float().mean().item()
print(f"학습 데이터 정확도: {train_accuracy:.1%}")

epoch  1 | loss = 1.1064
epoch 20 | loss = 0.0069
epoch 40 | loss = 0.0003
epoch 60 | loss = 0.0002
epoch 80 | loss = 0.0001
학습 데이터 정확도: 100.0%


### A-6. 한 문장의 의도 예측

텍스트 한 개를 정수 ID로 바꾼 뒤, 마지막 Hidden State로 예측한 의도 하나를 반환합니다.

In [6]:
@torch.inference_mode()
def predict_intent(model, text: str, token_to_id: dict, label_names: list[str]) -> str:
    # ① 문장을 공백 기준으로 나누어 토큰 ID로 변환합니다.
    token_ids = [token_to_id[token] for token in text.split()]
    x = torch.tensor([token_ids], dtype=torch.long)  # (배치=1, 시점 수)

    model.eval()
    # ② LSTM의 마지막 Hidden State를 활용한 클래스별 Logits를 얻습니다.
    logits, _, _, _ = model(x)
    # ③ 가장 높은 점수의 클래스 ID를 실제 의도 이름으로 바꿉니다.
    predicted_id = logits.argmax(dim=-1).item()
    return label_names[predicted_id]


example_texts_a = [
    "예약 내일 오후 시간 부탁",
    "취소 내일 오후 시간 부탁",
    "변경 내일 오후 시간 부탁",
]
for text in example_texts_a:
    print(f"{text} → {predict_intent(model_a, text, token_to_id_a, label_names_a)}")

예약 내일 오후 시간 부탁 → 예약
취소 내일 오후 시간 부탁 → 취소
변경 내일 오후 시간 부탁 → 변경


## B형 1단계 · LSTM 은닉 차원 변경 

A형 모델 구조를 참고해 **Hidden State 크기를 16에서 32로 변경**합니다. `head`의 입력 크기와 `forward()`의 연결을 직접 완성하세요.

- 목표: `output (2, 5, 32)`, `h_n (1, 2, 32)`, `c_n (1, 2, 32)`, `logits (2, 3)`
- 학습은 하지 않고 모델 구조와 Tensor Shape만 확인합니다.

### B-1. [학생 작성] 변경된 LSTM 모델

아래 클래스의 `TODO` 두 부분을 A형 코드를 참고해 완성하세요.

In [10]:
class StudentLSTMIntentClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, embedding_dim: int = 12, hidden_size: int = 32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # LSTM 입력 : (batch, sequence, 12)
        # LSTM 출력 : (batch, sequence, 32)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)

        # TODO 1: 마지막 Hidden State 차원(hidden_size)을 입력받는 Linear를 self.head에 정의하세요.
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, token_ids: torch.Tensor):
        embedded = self.embedding(token_ids)
        output, (h_n, c_n) = self.lstm(embedded)
        logits = self.head(h_n[-1])  # 마지막 계층·마지막 시점의 Hidden State 활용
        return logits, output, h_n, c_n

### B-2. 출력 Shape 점검 (제공 코드)

B-1을 완성한 뒤 실행해 A형과의 차원을 비교합니다.

In [11]:
model_b1 = StudentLSTMIntentClassifier(
    vocab_size=len(vocab_a), num_classes=len(label_names_a),
    embedding_dim=12, hidden_size=32,
)
model_b1.eval()
with torch.no_grad():
    logits_b1, output_b1, h_n_b1, c_n_b1 = model_b1(x_a[:2])

print("output:", tuple(output_b1.shape))
print("h_n:", tuple(h_n_b1.shape))
print("c_n:", tuple(c_n_b1.shape))
print("logits:", tuple(logits_b1.shape))

assert output_b1.shape == (2, 5, 32)
assert h_n_b1.shape == (1, 2, 32)
assert c_n_b1.shape == (1, 2, 32)
assert logits_b1.shape == (2, 3)
print("B형 1단계 통과")

output: (2, 5, 32)
h_n: (1, 2, 32)
c_n: (1, 2, 32)
logits: (2, 3)
B형 1단계 통과


## B형 2단계 · 요청 순서에 따른 최종 의도 분류 

A형에서는 문장 한 개의 의도를 예측했습니다. 이제는 **여러 문장을 한 번에 전달받아, 각 문장의 최종 의도와 예측 확률을 반환하는 함수**를 구현합니다.

예: `예약 했다가 나중에 취소` → 취소 / `취소 했다가 다시 예약` → 예약

이 단계의 모델은 **별도로 학습해 둔 가중치**를 사용합니다. 모델 학습 코드는 다시 작성하지 않습니다.

### B-3. 새로운 문장과 학습된 모델 준비

새 단어 사전과 두 개의 정답 클래스를 불러옵니다. 이 셀은 수정하지 않습니다.

In [ ]:
from pathlib import Path

weights_path = Path("weights/final_intent_lstm.pt")
if not weights_path.exists():
    raise FileNotFoundError("노트북과 같은 폴더에 weights/final_intent_lstm.pt 파일이 필요합니다.")

checkpoint_b2 = torch.load(weights_path, map_location="cpu", weights_only=True)
vocab_b2 = checkpoint_b2["vocab"]
token_to_id_b2 = {token: idx for idx, token in enumerate(vocab_b2)}
label_names_b2 = checkpoint_b2["labels"]

# 학습 데이터·어휘·클래스가 A형과 다르므로 별도 모델을 생성합니다.
model_b2 = LSTMIntentClassifier(
    vocab_size=len(vocab_b2), num_classes=len(label_names_b2),
    embedding_dim=checkpoint_b2["embedding_dim"],
    hidden_size=checkpoint_b2["hidden_size"],
)
model_b2.load_state_dict(checkpoint_b2["state_dict"])
model_b2.eval()

print("제공 가중치 로드 완료")
print("B형 2단계 클래스:", label_names_b2)
print("B형 2단계 Hidden State 차원:", checkpoint_b2["hidden_size"])
print("처리 방식: 문장별 추론 → 서로 다른 길이의 문장도 처리 가능")

### B-4. [학생 작성] 여러 문장의 최종 의도 예측 함수

A형의 `predict_intent()`를 참고해 아래 함수를 완성하세요.

**입력:** 문장 리스트 `sentences` / **반환:** `(원문, 예측 의도, 예측 확률)` 튜플의 리스트

필수 구현: **문장 반복 처리 → Token ID 변환(모르는 단어는 `<UNK>`) → LSTM 추론 → Softmax·Argmax → 결과 저장**. 문장 길이가 다르므로 각 문장을 배치 크기 1로 처리합니다.

In [ ]:
def predict_final_intent(
    model, sentences: list[str], token_to_id: dict, label_names: list[str]
) -> list[tuple[str, str, float]]:

    # TODO 1 [A형 복습]
    # A-6에서 사용했던 model.eval()을 작성하세요.
    # 여러 문장의 예측 결과를 담을 빈 리스트 results도 만드세요.


    # 여러 문장을 하나씩 처리합니다.
    # 문장마다 길이가 다르므로 Batch Size=1로 추론합니다.
    for text in sentences:

        # TODO 2 [새로운 내용 - 코드 제공]
        # A형에서는 token_to_id[token]으로 변환했습니다.
        # 이번에는 학습 어휘에 없는 단어가 등장할 수도 있습니다.
        # dict.get()을 사용하면 없는 단어를 <UNK> ID로 대체할 수 있습니다.
        token_ids = [
            token_to_id.get(token, token_to_id["<UNK>"])
            for token in text.split()
        ]

        # TODO 3 [A형 복습]
        # A-6에서 했던 것처럼:
        # ① token_ids를 (1, 시점 수) Shape의 Tensor로 만드세요.
        #    힌트: torch.tensor([token_ids], dtype=torch.long)
        # ② model(x)를 실행해 logits를 받으세요.
        #    모델은 4개의 값을 반환한다는 점을 기억하세요.


        # TODO 4 [새로운 내용 - Softmax 코드 제공]
        # Logits는 클래스별 점수이므로 Softmax로 확률로 변환합니다.
        # Shape: (1, 클래스 수)
        probabilities = torch.softmax(logits, dim=-1)

        # [학생 작성 - A형 복습]
        # A-6에서 사용한 argmax(dim=-1).item()을 활용하세요.
        # 이번에는 probabilities에서 가장 높은 확률의
        # 클래스 ID를 predicted_id에 저장하면 됩니다.


        # TODO 5 [새로운 내용 - 확률 추출 코드 제공]
        # 0은 첫 번째(유일한) 문장을 의미합니다.
        # predicted_id에 해당하는 클래스의 확률을 가져옵니다.
        confidence = float(probabilities[0, predicted_id].item())

        # [학생 작성 - A형 복습 및 응용]
        # ① A-6처럼 label_names[predicted_id]로 클래스 이름을 찾으세요.
        # ② results.append()를 이용해 아래 세 값을 저장하세요.
        #    (원본 문장 text, 예측 클래스 이름, confidence)


    # TODO 5 마무리
    # 모든 문장의 결과가 담긴 results를 반환하세요.

    raise NotImplementedError("B-4를 완성하세요.")


### B-5. 새로운 문장 예측 결과 확인 

B-4를 완성한 뒤 실행하세요. 문장의 **요청 순서가 바뀌면 최종 의도도 바뀌는지** 비교합니다.

In [ ]:
# 아래 문장은 제공 가중치의 학습 문장과 동일하지 않은 확인용 문장입니다.
test_cases_b2 = [
    ("예약 했다가 나중에 취소", "취소"),
    ("취소 했다가 다시 예약", "예약"),
    ("예약 하고 그대로 진행", "예약"),
    ("취소 하고 그대로 유지", "취소"),
    ("처음 예약 했다가 결국 취소", "취소"),
    ("처음 취소 했다가 나중에 예약", "예약"),
    ("예약 했다가 취소 했다가 예약", "예약"),
    ("취소 했다가 예약 했다가 취소", "취소"),
]

results_b2 = predict_final_intent(
    model_b2, [text for text, _ in test_cases_b2], token_to_id_b2, label_names_b2
)
assert len(results_b2) == len(test_cases_b2), "문장별 결과를 빠짐없이 반환해야 합니다."

correct_b2 = 0
for (text, expected), (input_text, prediction, probability) in zip(test_cases_b2, results_b2):
    assert input_text == text, "원문 순서를 유지해야 합니다."
    assert isinstance(probability, (float, int)) and 0 <= probability <= 1
    correct_b2 += prediction == expected
    print(f"{text:<30} | 정답: {expected} | 예측: {prediction} | 확률: {probability:.3f}")

print()
print(f"확인 문장 정확도: {correct_b2}/{len(test_cases_b2)}")
assert correct_b2 == len(test_cases_b2), "Token ID 변환, 마지막 Hidden State, 예측 함수 내용을 확인하세요."
print("B형 2단계 통과")

### 마무리 확인

- **RNN 언어모델(실습 1):** 각 시점의 출력으로 다음 단어를 예측했습니다.
- **LSTM 문장 분류(실습 2):** 마지막 Hidden State로 문장 전체의 의도를 예측했습니다.
- LSTM은 `h_n`뿐 아니라 `c_n`도 계산하며, 문장 분류에는 `h_n[-1]`을 사용했습니다.